# Book Recommendation System using Collaborative Filtering

This notebook implements an **Item-Based Collaborative Filtering** system to recommend books. We use a dataset containing book details and user ratings to calculate book similarities using the **Cosine Similarity** metric via the K-Nearest Neighbors (KNN) algorithm.

## 1. Import Libraries

We start by importing the required Python libraries for data processing and machine learning.

In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
print("Libraries successfully imported!")

Libraries successfully imported!


## 2. Load the Data

We load `Books.csv` and `Ratings.csv` using pandas from the current working directory.

In [2]:
# Load datasets
books = pd.read_csv('Books.csv', low_memory=False)
ratings = pd.read_csv('Ratings.csv', low_memory=False)

# Display dataset shapes and columns
print(f"Books dataset shape: {books.shape}")
print(f"Ratings dataset shape: {ratings.shape}")
print("\nBooks columns:", books.columns.tolist())
print("Ratings columns:", ratings.columns.tolist())

Books dataset shape: (19784, 8)
Ratings dataset shape: (276068, 3)

Books columns: ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']
Ratings columns: ['User-ID', 'ISBN', 'Book-Rating']


## 3. Data Cleaning and Filtering

To handle matrix sparsity and keep memory usage low, we:
1. Drop missing values.
2. Filter out users who have rated fewer than 50 books.
3. Filter out books that have received fewer than 10 ratings.

In [3]:
# Drop missing values
books.dropna(inplace=True)
ratings.dropna(inplace=True)

# Filter users who have given at least 50 ratings
user_counts = ratings['User-ID'].value_counts()
ratings = ratings[ratings['User-ID'].isin(user_counts[user_counts >= 50].index)]

# Filter books that have at least 10 ratings
book_counts = ratings['ISBN'].value_counts()
ratings = ratings[ratings['ISBN'].isin(book_counts[book_counts >= 10].index)]

print(f"Filtered Ratings dataset shape: {ratings.shape}")

Filtered Ratings dataset shape: (32142, 3)


## 4. Data Wrangling & Creating the User-Item Matrix

We merge our datasets on the book ID (`ISBN`), pivot the table so that rows represent `Book-Title` and columns represent `User-ID`, fill missing ratings with 0, and convert it to a sparse matrix for efficient computation.

In [4]:
# Merge ratings and books
merged_df = pd.merge(ratings, books, on='ISBN')

# Drop duplicates to ensure a clean pivot table
merged_df.drop_duplicates(['User-ID', 'Book-Title'], inplace=True)

# Create the user-item matrix (pivot table)
pivot_matrix = merged_df.pivot(index='Book-Title', columns='User-ID', values='Book-Rating').fillna(0)

# Convert pivot table to a sparse matrix
sparse_features = csr_matrix(pivot_matrix.values)

print(f"User-Item pivot matrix shape (Books x Users): {pivot_matrix.shape}")

User-Item pivot matrix shape (Books x Users): (1418, 762)


## 5. Model Implementation

We use scikit-learn's `NearestNeighbors` algorithm with the `cosine` similarity metric to identify similar books.

In [5]:
# Initialize and fit Nearest Neighbors model
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(sparse_features)
print("Model training complete!")

Model training complete!


## 6. Output & Evaluation (Recommendations)

We write a helper function to recommend similar books. Given a book title, it searches the user-item pivot matrix, finds the nearest neighbors, and outputs the top 5 matches.

In [6]:
def recommend_books(book_title):
    # Check if book title is in the pivot matrix index
    matching_books = [title for title in pivot_matrix.index if book_title.lower() in title.lower()]
    if not matching_books:
        print(f"Book '{book_title}' not found in the matrix.")
        return

    # Select the first match
    selected_title = matching_books[0]
    print(f"Showing recommendations for: {selected_title}\n")

    # Get the index of the matching book
    book_idx = pivot_matrix.index.get_loc(selected_title)

    # Query Nearest Neighbors
    distances, indices = model.kneighbors(
        pivot_matrix.iloc[book_idx, :].values.reshape(1, -1),
        n_neighbors=6
    )

    # Print the top 5 recommendations (skipping the first one as it is the query book itself)
    count = 1
    for i in range(1, len(indices.flatten())):
        recommended_book = pivot_matrix.index[indices.flatten()[i]]
        print(f"{count}: {recommended_book} (Distance: {distances.flatten()[i]:.4f})")
        count += 1

# Example run
recommend_books("Harry Potter and the Sorcerer's Stone")

Showing recommendations for: Harry Potter and the Sorcerer's Stone (Book 1)

1: Harry Potter and the Goblet of Fire (Book 4) (Distance: 0.6121)
2: The Hobbit: or There and Back Again (Distance: 0.6124)
3: Harry Potter and the Chamber of Secrets (Book 2) (Distance: 0.6353)
4: N Is for Noose (Kinsey Millhone Mysteries (Hardcover)) (Distance: 0.6567)
5: Harry Potter and the Prisoner of Azkaban (Book 3) (Distance: 0.6821)
